# Utilizing Fuzzy Reinforcement Learning for Volatile Stock Market Prediction

## 1. Core Methodology: FLSPI Overview
The paper introduces **Fuzzy Least Squares Policy Iteration (FLSPI)** to address the complex, volatile, and non-stationary nature of financial data. FLSPI merges the strengths of:
* **Fuzzy Systems:** Manages the uncertainty and high dimensionality of the stock market through smooth state transitions.
* **Least Squares Policy Iteration (LSPI):** Provides robust value function approximation for reinforcement learning (RL) in continuous spaces.

Instead of predicting exact discrete states, FLSPI calculates the continuous action (predicted price movement) by summing selected basis actions, weighted by the firing strength of fuzzy rules:
$$a_{t}(s_{t})=\sum_{i=1}^{R}\mu_{i}(s_{t})a_{i^{+}}$$

## 2. Algorithmic Design & Hyperparameter Tuning
To tailor FLSPI for financial time-series, the authors implemented several domain-specific optimizations:

* **State Space Formulation:** The most effective configuration uses a two-dimensional state space consisting of the **Close Price** and the **MACD** (Moving Average Convergence Divergence) indicator.
* **Data-Driven Fuzzy Partitioning:** Instead of rigid boundaries, the model uses **Fuzzy C-means clustering** to define Gaussian membership functions. This allows for smooth transitions between market states, preventing abrupt policy changes.
* **Action Space & Basis Actions:** The continuous action space (price trends) is approximated using a fixed set of $m=11$ basis actions. These are generated symmetrically based on the mean and standard deviation of historical differenced data (e.g., within $[\mu-\sigma, \mu+\sigma]$ for short-term predictions).
* **Dynamic Discount Factor:** Unlike standard RL models that use a fixed discount factor ($\gamma$), this model dynamically adjusts it based on historical market volatility. During high volatility, $\gamma$ approaches zero; during stable periods, it approaches 0.9:
$$\gamma=\frac{0.9}{\text{historical volatility} + 1}$$
* **Reward Function:** The agent is rewarded for minimizing squared error, scaled to encourage precision and penalize large deviations quadratically:
$$r=\frac{10}{1+(s_{predicted}-s_{real})^{2}}$$

## 3. Computational Efficiency Enhancements
To mitigate the high computational cost of standard LSPI matrix operations, the algorithm exploits **matrix sparsity**. By updating only the non-zero elements associated with active basis functions, the computation time is drastically reduced, allowing for faster and more efficient weight vector adjustments.

---

## 4. Application & Experimental Setup
The methodology was rigorously tested against real-world financial data to evaluate both statistical accuracy and practical trading viability.

* **Datasets:** 10 major global stocks (including AAPL, MSFT, NVDA, TSLA, JPM).
* **Prediction Horizons:** Short-term (1 to 30 days) and Long-term (60 to 150 days).
* **Baselines for Comparison:** * Traditional Statistical: **ARIMA**
    * Deep Learning: **LSTM**
    * Foundation Models: **Chronos-Bolt-Tiny** (Zero-shot and Few-shot variants)
* **Evaluation Metrics:** * *Statistical:* RMSE, MAE, MAPE, MASE.
    * *Financial/Directional:* Hit Rate, Balanced Accuracy, Sharpe Ratio, Maximum Drawdown (MDD).

## 5. Key Findings and Performance
* **Superior Accuracy:** The multivariate FLSPI model consistently achieved the lowest RMSE, MAE, and MAPE across most prediction horizons compared to ARIMA and LSTM.
* **Robustness in Volatility:** While traditional models (like ARIMA) degraded rapidly over longer forecasting horizons, FLSPI maintained stable and reliable predictions.
* **Financial Viability:** Beyond statistical accuracy, FLSPI demonstrated strong directional reliability (Hit Rate) and provided the most balanced trade-off between risk and reward (Sharpe/Calmar ratios) in a simulated transaction-cost-aware trading environment.
* **Competitive Landscape:** Statistical tests (Friedman and Nemenyi post-hoc) confirmed that FLSPI's performance is grouped alongside the fine-tuned (few-shot) Chronos foundation model as the top-performing methodologies in the study.

## Define the State, Action, and Reward.

 ### The State Space ($S$): Instead of Stock Price and MACD, model needs to understand the current consumption landscape. 
 
 ### A great 2D state space for this would be:

* Current Monthly Consumption: The raw or normalized volume/sales.

* Momentum Seasonality: The difference between this month and the previous month (Month-over-Month change), or the difference between this month and the same month last year (to capture seasonal holidays like Thanksgiving or Christmas).

### The Action Space ($A$): 

The algorithm's "action" is its prediction of the change in consumption for the next month.

 * calculate the historical mean and standard deviation of month-to-month changes in the data.

 * generate 11 "Basis Actions" evenly spaced from $[Mean - 2\times Std]$ to $[Mean + 2\times Std]$.

### The Reward Function ($R$): 

Use the paper's exact reward function. If the algorithm's chosen combination of basis actions predicts the next month's consumption perfectly, it gets a high reward (approaching 10). If it misses, the reward drops to near 0.

### Fuzzy Partitions: 

Divide the "Current Consumption" into 3 fuzzy states (Low, Medium, High) and "Momentum" into 2 fuzzy states (Falling, Rising). This gives us $3 \times 2 = 6$ fuzzy rules.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib # 日本語フォント対応
import seaborn as sns

In [5]:
# df.to_csv('/Users/balalala/Documents/GitHub/Consumption_Data/Data_time_series/en_setai_over2_monthly.csv', index=False)
df_share = pd.read_csv('/Users/balalala/Documents/GitHub/Consumption_Data/Data_time_series/en_share_over2_monthly.csv')

In [6]:
# Filter for rows where Medium (中分類) and Minor (小分類) are '-'
# This isolates the top-level "Major" rows

major_rows = df[ (df['Category level 1'] != '-') & (df['Category level 2'] == '-') & (df['Category level 3'] == '-') ]

major_dict = dict(zip(major_rows['Category level 1'], major_rows['Item']))
print(major_dict['1']) # Output: 食料

hierarchy = {}

# Iterate through every row
for index, row in df.iterrows():
    major_id = row['Category level 1']
    med_id = row['Category level 2']
    min_id = row['Category level 3']
    name = row['Item']

    # Skip header/garbage rows if any
    if major_id == '-': continue 

    # 1. Initialize Major Level
    # We use the ID as the key, but store the Name inside
    if major_id not in hierarchy:
        # Look up the major name from our simple dict
        major_name = major_dict.get(major_id, "他")
        hierarchy[major_id] = {'name': major_name, 'med': {}}

    # 2. Add Medium Level (if this row represents a Medium category or deeper)
    if med_id != '-':
        if med_id not in hierarchy[major_id]['med']:
             # Use the name if this is the defining row, otherwise generic placeholder until found
            hierarchy[major_id]['med'][med_id] = {'name': name, 'small': {}}
        
        # Update name if this is exactly the Medium definition row
        if min_id == '-':
            hierarchy[major_id]['med'][med_id]['name'] = name

    # 3. Add Minor Level (if this row represents a Minor category)
    if min_id != '-':
        # Add the minor category to the medium's children
        hierarchy[major_id]['med'][med_id]['small'][min_id] = name
# Usage:
# hierarchy[1]['children'][1]['name']  -> Access the name of Medium Category 1 inside Major 1
# Initialize the flat dictionary
# Key = Item Name (e.g., 'パン'), Value = Formatted String (e.g., '食料ーパン')

name_to_path_map = {}

# Iterate through the hierarchy
for major_id, major_data in hierarchy.items():
    major_name = major_data['name']
    
    # 1. Add the Major category itself (Optional, if needed)
    # name_to_path_map[major_name] = major_name
    
    # Check if 'med' exists
    if 'med' in major_data:
        for med_id, med_data in major_data['med'].items():
            med_name = med_data['name']
            
            # 2. Add Medium Category: Input '穀類' -> Output '食料ー穀類'
            name_to_path_map[med_name] = f"{major_name}ー{med_name}"
            
            # Check if 'small' exists
            if 'small' in med_data:
                for small_id, small_name in med_data['small'].items():
                    # 3. Add Small Category: Input 'パン' -> Output '食料ーパン'
                    # Note: Using strict user format "Major-Small". 
                    # If you wanted full path "Major-Med-Small", use: f"{major_name}ー{med_name}ー{small_name}"
                    name_to_path_map[small_name] = f"{major_name}ー{med_name}ー{small_name}"

# Check the result
print(name_to_path_map['bread']) 
# Output: '食料ーパン'

def get_category_path(item_name):
    return name_to_path_map.get(item_name, item_name) # Returns "他" if not found

print(get_category_path('bread'))

# --- Step 1: Flatten your hierarchy dictionary ---
# This converts: {'1': {'name': 'food', 'med': {'1': {'name': 'grains', 'small': {'1': 'rice'...
# To: {'rice': 'food', 'bread': 'food', ...}

category_map = {}

for l1_id, l1_info in hierarchy.items():
    l1_name = l1_info['name']
    for l2_id, l2_info in l1_info['med'].items():
        # Option A: Map to Level 1 (Food vs Transport)
        # Option B: Map to Level 2 (Grains vs Meat)
        # Let's map to Level 1 for a high-level systemic view
        for l3_id, l3_name in l2_info['small'].items():
            category_map[l3_name] = l1_name

# Convert to a Series for easy grouping
category_series = pd.Series(category_map)
category_series

food
foodーgrainsーbread
foodーgrainsーbread


rice                                                                        food
bread                                                                       food
noodles                                                                     food
other grains                                                                food
fresh seafood                                                               food
salted and dried seafood                                                    food
Fish paste products                                                         food
Other processed seafood products                                            food
fresh meat                                                                  food
processed meat                                                              food
milk                                                                        food
dairy products                                                              food
egg                         

In [7]:
# Dataframe for 3 levels of category hierarchy

c1 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']=='-') & (df_share['Category level 3']=='-')]

c2 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']!='-') & (df_share['Category level 3']=='-')]

c3 = df_share[(df_share['Category level 1']!='-') & (df_share['Category level 2']!='-') & (df_share['Category level 3']!='-')]

In [10]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from scipy.stats import zscore

def prep_data_for_stl(raw_df: pd.DataFrame, meta_col_count: int = 3) -> pd.DataFrame:
    """
    Cleans raw DataFrame and transposes it into a Time x Items format.
    """
    # 1. Extract and Transpose
    df_ts = raw_df.set_index('Item').iloc[:, meta_col_count:]
    df_t = df_ts.T

    # 2. Index Formatting
    df_t.index = pd.to_datetime(df_t.index)
    df_t = df_t.sort_index()

    # 3. Type Casting & Handle Missing Data (STL requires no NaNs)
    df_t = df_t.astype(np.float64)
    # df_t = df_t.ffill().bfill() # Forward/backward fill any missing percentages
    
    return df_t

def get_residuals(series: pd.Series, period: int = 12) -> pd.Series:
    """
    Extracts the residual component from a time series using STL.
    """
    # robust=True handles outliers (like COVID-19 anomalies) better
    res = STL(series, period=period, robust=True).fit()
    return res.resid


In [11]:
# STEP 1: Clean and Transpose raw consumption data
df_clean = prep_data_for_stl(c1, meta_col_count=3)

# STEP 2: Apply STL to get the Residuals (Deviations from trend/seasonality)
df_residuals = df_clean.apply(get_residuals, period=12)

# STEP 3: Apply Z-Score Normalization
df_res_norm = df_residuals.apply(zscore)

# STEP 4: Output as Markdown
# Using .to_markdown() converts the dataframe head into a formatted markdown table
markdown_output = df_res_norm.head().to_markdown()
print(markdown_output)
print(df_res_norm.shape)

|                     |         food |   residence |   Utilities/Water |   Furniture/household supplies |   clothing and footwear |   health care |   Transportation/Communication |   education |   educational entertainment |   Other consumption expenditure |
|:--------------------|-------------:|------------:|------------------:|-------------------------------:|------------------------:|--------------:|-------------------------------:|------------:|----------------------------:|--------------------------------:|
| 2000-01-01 00:00:00 |  0.242659    |  0.146333   |        -0.303383  |                     -0.0391812 |                0.760769 |      0.524308 |                     -0.241736  |   0.20041   |                 -0.00815925 |                      -0.0150675 |
| 2000-02-01 00:00:00 | -0.000365243 | -0.0705798  |         0.0480249 |                     -0.334306  |                0.15343  |      0.630427 |                      0.0373436 |  -0.294522  |                  0.370517   

In [12]:
df_clean

Item,food,residence,Utilities/Water,Furniture/household supplies,clothing and footwear,health care,Transportation/Communication,education,educational entertainment,Other consumption expenditure
2000-01-01,23.7645,5.3482,8.0602,3.1668,6.3310,3.5337,10.1369,4.0475,10.2322,25.3791
2000-02-01,25.2213,6.3489,8.8343,3.1679,4.9542,4.0559,10.6794,4.9821,10.2307,21.5249
2000-03-01,23.7746,5.4873,7.5544,3.2960,5.4768,3.5090,11.6589,5.2809,10.7383,23.2241
2000-04-01,23.0688,5.6121,6.8332,2.8773,5.3771,3.3557,12.3349,7.1729,10.1844,23.1839
2000-05-01,26.3850,6.2392,6.8300,3.6543,6.0642,3.6734,11.6957,3.7308,10.9617,20.7661
...,...,...,...,...,...,...,...,...,...,...
2025-07-01,30.6293,6.5834,6.5994,5.2876,3.2840,5.4708,15.2479,2.5918,10.0146,14.2914
2025-08-01,32.6275,6.0243,6.6855,4.4971,2.6648,4.9488,13.5577,2.3511,11.9098,14.7332
2025-09-01,30.7156,5.0565,7.3117,4.0285,2.4171,5.2847,15.5883,4.0189,10.1206,15.4584
2025-10-01,30.7464,5.9598,7.0632,3.8423,3.3085,5.5130,13.8462,4.4246,10.1596,15.1363
